# `remove_words` — What Changed

This notebook walks through all the changes made to the `remove_words` wrangle.

## Summary of changes

| Parameter | Before | After |
|---|---|---|
| `tokenize_to_remove` default | `False` | **`True`** |
| `ignore_case` default | `True` | `True` *(no change, now documented)* |
| `characters_to_consider` | *(did not exist)* | **new** — `letters_numbers_only` (default) or `all` |

**Behaviour rules (always apply when `tokenize_to_remove=True`):**
- Multiple consecutive spaces are collapsed to a single space before tokenizing
- Leading/trailing punctuation is stripped from both `input` and `to_remove` values before tokenizing
- If a `to_remove` value is already a **list**, each element is treated as an individual token — it is **not** re-split by whitespace
- If `input` is already a **list**, each element is treated as an individual token — it is not re-split

In [1]:
import pandas as pd
import wrangles

def run(data: dict, recipe_body: str) -> pd.DataFrame:
    """Helper: run a recipe snippet and return the dataframe."""
    recipe = f"""
    wrangles:
    {recipe_body}
    """
    return wrangles.recipe.run(recipe, dataframe=pd.DataFrame(data))

---
## 1. `tokenize_to_remove` now defaults to `True`

**Before:** you had to explicitly set `tokenize_to_remove: True` to split a `to_remove` string like
`'Metal Carbon'` into individual tokens `['Metal', 'Carbon']`.

**After:** this is the default. The string is split by whitespace automatically.

In [2]:
# 'Metal Carbon' in to_remove is split into 'Metal' and 'Carbon' — both removed individually
df = run(
    {'col': ['Metal Carbon Water Tank'], 'rem': ['Metal Carbon']},
    """
    - remove_words:
        input: col
        to_remove:
          - rem
        output: out
    """
)
df[['col', 'rem', 'out']]

INFO:root:: Wrangling :: remove_words :: col >> out


,col,rem,out
0,Metal Carbon Water Tank,Metal Carbon,Water Tank


In [3]:
# Use tokenize_to_remove: False when you want to match multi-word PHRASES exactly
df = run(
    {'col': ['Metal Carbon Water Tank'], 'rem': ['Metal Carbon']},
    """
    - remove_words:
        input: col
        to_remove:
          - rem
        output: out
        tokenize_to_remove: False
    """
)
# 'Metal Carbon' is treated as a two-word phrase — no match for just 'Metal' or 'Carbon'
# but the phrase 'Metal Carbon' IS present, so it is removed
df[['col', 'rem', 'out']]

INFO:root:: Wrangling :: remove_words :: col >> out


,col,rem,out
0,Metal Carbon Water Tank,Metal Carbon,Water Tank


---
## 2. `characters_to_consider` — new parameter

| Value | Behaviour |
|---|---|
| `letters_numbers_only` *(default)* | Strip all non-alphanumeric characters from tokens **before comparing**. A token like `A-100` is compared as `A100`. |
| `all` | Use the full token for comparison. `A-100` ≠ `A100`. |

> **Important:** stripping is only for comparison. The original text tokens that are *not* removed are preserved as-is in the output.

In [4]:
# letters_numbers_only (default): 'A-100' in input matches 'A100' in to_remove
df = run(
    {'col': ['A-100 B-200 C-300'], 'rem': ['A100 C300']},
    """
    - remove_words:
        input: col
        to_remove:
          - rem
        output: out
        # characters_to_consider: letters_numbers_only  # this is the default
    """
)
df[['col', 'rem', 'out']]  # expect: 'B-200'

INFO:root:: Wrangling :: remove_words :: col >> out


,col,rem,out
0,A-100 B-200 C-300,A100 C300,B-200


In [5]:
# characters_to_consider: all — full token used, 'A-100' != 'A100', nothing removed
df = run(
    {'col': ['A-100 B-200 C-300'], 'rem': ['A100 C300']},
    """
    - remove_words:
        input: col
        to_remove:
          - rem
        output: out
        characters_to_consider: all
    """
)
df[['col', 'rem', 'out']]  # expect: 'A-100 B-200 C-300' (unchanged)

INFO:root:: Wrangling :: remove_words :: col >> out


,col,rem,out
0,A-100 B-200 C-300,A100 C300,A-100 B-200 C-300


### Practical example: part numbers with mixed punctuation

In [6]:
df = run(
    {
        'description': [
            'M6-1.0 x 20mm Hex Cap Screw, Stainless',
            'M8x1.25 Socket Head Cap Screw, Grade-8',
            '1/4"-20 UNC Hex Bolt, Zinc Plated',
        ],
        'to_remove': [
            ['Stainless', 'Zinc', 'Grade8'],   # note 'Grade8' vs 'Grade-8'
            ['Stainless', 'Zinc', 'Grade8'],
            ['Stainless', 'Zinc', 'Grade8'],
        ]
    },
    """
    - remove_words:
        input: description
        to_remove:
          - to_remove
        output: cleaned
    """
)
df[['description', 'cleaned']]

INFO:root:: Wrangling :: remove_words :: description >> cleaned


,description,cleaned
0,"M6-1.0 x 20mm Hex Cap Screw, Stainless","M6-1.0 x 20mm Hex Cap Screw,"
1,"M8x1.25 Socket Head Cap Screw, Grade-8","M8x1.25 Socket Head Cap Screw,"
2,"1/4""-20 UNC Hex Bolt, Zinc Plated","1/4""-20 UNC Hex Bolt, Plated"


---
## 3. Punctuation & spaces stripped from ends before tokenizing

Leading/trailing punctuation is stripped from the **full input string** and from **each `to_remove` value** before they are tokenized.  
This handles common data quality issues like trailing `?`, leading `$`, extra spaces, etc.

In [7]:
# Trailing '?' on input and leading '?' / trailing '!' on to_remove — still match
df = run(
    {'col': ['Any, Words, Overlap?'], 'rem': ['Any Words Overlap']},
    """
    - remove_words:
        input: col
        to_remove:
          - rem
        output: out
    """
)
df[['col', 'rem', 'out']]  # expect: '' (empty string — all tokens matched)

INFO:root:: Wrangling :: remove_words :: col >> out


,col,rem,out
0,"Any, Words, Overlap?",Any Words Overlap,


In [8]:
# Messy whitespace and punctuation on to_remove values
df = run(
    {'col': ['  Alpha Beta Gamma  '], 'rem': ['?Alpha! Beta,']},
    """
    - remove_words:
        input: col
        to_remove:
          - rem
        output: out
    """
)
df[['col', 'rem', 'out']]  # expect: 'Gamma'

INFO:root:: Wrangling :: remove_words :: col >> out


,col,rem,out
0,Alpha Beta Gamma,"?Alpha! Beta,",Gamma


---
## 4. Multiple spaces collapsed before tokenizing

Multiple consecutive spaces in a `to_remove` string are reduced to one before splitting, so tokens never accidentally contain a space.

In [9]:
# 'Metal  Carbon' (double space) → treated the same as 'Metal Carbon'
df = run(
    {'col': ['Metal Carbon Water Tank'], 'rem': ['Metal  Carbon']},
    """
    - remove_words:
        input: col
        to_remove:
          - rem
        output: out
    """
)
df[['col', 'rem', 'out']]  # expect: 'Water Tank'

INFO:root:: Wrangling :: remove_words :: col >> out


,col,rem,out
0,Metal Carbon Water Tank,Metal Carbon,Water Tank


---
## 5. List inputs skip the tokenize step

If a `to_remove` column already contains a **list**, each element is treated as one individual token — it is **not** split further by whitespace.  
Similarly, if the `input` column contains a list, each element is treated as a token directly.

This means a multi-word list element like `'3/8" x 7/16"'` is treated as a single unit, not three words.

In [10]:
# Input is a list — each element is already a token
df = run(
    {
        'parts': [['Steel', 'Blue', 'Bottle']],
        'materials': [['Steel']],
        'colours': [['Blue']],
    },
    """
    - remove_words:
        input: parts
        to_remove:
          - materials
          - colours
        output: product
    """
)
df[['parts', 'product']]  # expect: 'Bottle'

INFO:root:: Wrangling :: remove_words :: parts >> product


,parts,product
0,"[Steel, Blue, Bottle]",Bottle


In [11]:
# to_remove list element contains spaces — treated as ONE token, NOT re-tokenized
# Use tokenize_to_remove: False if you need phrase-based removal of multi-word list items
df = run(
    {
        'col': ['Brand T18 Round 3/8" x 7/16" Pack'],
        'rem': [['3/8" x 7/16"']],  # list element with spaces
    },
    """
    - remove_words:
        input: col
        to_remove:
          - rem
        output: tokenize_true
        tokenize_to_remove: True   # default — list item NOT re-tokenized, stays in output
    """
)
df2 = run(
    {
        'col': ['Brand T18 Round 3/8" x 7/16" Pack'],
        'rem': [['3/8" x 7/16"']],
    },
    """
    - remove_words:
        input: col
        to_remove:
          - rem
        output: tokenize_false
        tokenize_to_remove: False  # phrase-based — the whole '3/8" x 7/16"' IS removed
    """
)
pd.concat([df[['col', 'tokenize_true']], df2[['tokenize_false']]], axis=1)

INFO:root:: Wrangling :: remove_words :: col >> tokenize_true
INFO:root:: Wrangling :: remove_words :: col >> tokenize_false


,col,tokenize_true,tokenize_false
0,"Brand T18 Round 3/8"" x 7/16"" Pack",Brand T18 Round Pack,Brand T18 Round Pack


---
## 6. `ignore_case` — defaults to `True` (no change, now documented)

The default was already `True`. It is now clearly stated in the schema.

In [12]:
# ignore_case: True (default) — case-insensitive matching
df = run(
    {'col': ['METAL carbon Water TANK'], 'rem': ['metal CARBON']},
    """
    - remove_words:
        input: col
        to_remove:
          - rem
        output: out
        ignore_case: True
    """
)
df[['col', 'rem', 'out']]  # expect: 'Water TANK'

INFO:root:: Wrangling :: remove_words :: col >> out


,col,rem,out
0,METAL carbon Water TANK,metal CARBON,Water TANK


In [13]:
# ignore_case: False — case-sensitive, only exact case matches are removed
df = run(
    {'col': ['METAL carbon Water TANK'], 'rem': ['metal CARBON']},
    """
    - remove_words:
        input: col
        to_remove:
          - rem
        output: out
        ignore_case: False
    """
)
# 'metal' != 'METAL', 'CARBON' != 'carbon' → nothing removed
df[['col', 'rem', 'out']]

INFO:root:: Wrangling :: remove_words :: col >> out


,col,rem,out
0,METAL carbon Water TANK,metal CARBON,METAL carbon Water TANK


---
## 7. Parameter combinations reference

The table below shows how `tokenize_to_remove`, `ignore_case`, and `characters_to_consider` interact.

Input: `'STEEL A-100 Bottle'`  
to_remove: `'steel A100'`

In [14]:
import itertools

combos = list(itertools.product(
    [True, False],           # tokenize_to_remove
    [True, False],           # ignore_case
    ['letters_numbers_only', 'all'],  # characters_to_consider
))

rows = []
for tokenize, ic, ctc in combos:
    df = run(
        {'col': ['STEEL A-100 Bottle'], 'rem': ['steel A100']},
        f"""
        - remove_words:
            input: col
            to_remove:
              - rem
            output: out
            tokenize_to_remove: {str(tokenize)}
            ignore_case: {str(ic)}
            characters_to_consider: {ctc}
        """
    )
    rows.append({
        'tokenize_to_remove': tokenize,
        'ignore_case': ic,
        'characters_to_consider': ctc,
        'output': repr(df['out'].iloc[0]),
    })

pd.DataFrame(rows)

INFO:root:: Wrangling :: remove_words :: col >> out
INFO:root:: Wrangling :: remove_words :: col >> out
INFO:root:: Wrangling :: remove_words :: col >> out
INFO:root:: Wrangling :: remove_words :: col >> out
INFO:root:: Wrangling :: remove_words :: col >> out
INFO:root:: Wrangling :: remove_words :: col >> out
INFO:root:: Wrangling :: remove_words :: col >> out
INFO:root:: Wrangling :: remove_words :: col >> out


,tokenize_to_remove,ignore_case,characters_to_consider,output
0,True,True,letters_numbers_only,'Bottle'
1,True,True,all,'A-100 Bottle'
2,True,False,letters_numbers_only,'STEEL Bottle'
3,True,False,all,'STEEL A-100 Bottle'
4,False,True,letters_numbers_only,'Bottle'
5,False,True,all,'STEEL A-100 Bottle'
6,False,False,letters_numbers_only,'STEEL A-100 Bottle'
7,False,False,all,'STEEL A-100 Bottle'


---
## 8. Real-world example: cleaning product descriptions

A common use-case — strip material names, colour names, and noise words from a description to extract the core product noun.

In [15]:
data = {
    'description': [
        'Steel Blue Bottle',
        'Aluminum Red Can',
        'Rubber Yellow Tire',
        'Titanium Blue Pipe',
    ],
    'materials': [
        ['Steel', 'Aluminum', 'Rubber', 'Titanium', 'Iron'],
        ['Steel', 'Aluminum', 'Rubber', 'Titanium', 'Iron'],
        ['Steel', 'Aluminum', 'Rubber', 'Titanium', 'Iron'],
        ['Steel', 'Aluminum', 'Rubber', 'Titanium', 'Iron'],
    ],
    'colours': [
        ['Blue', 'Red', 'Yellow', 'Brown'],
        ['Blue', 'Red', 'Yellow', 'Brown'],
        ['Blue', 'Red', 'Yellow', 'Brown'],
        ['Blue', 'Red', 'Yellow', 'Brown'],
    ],
}

df = run(
    data,
    """
    - remove_words:
        input: description
        to_remove:
          - materials
          - colours
        output: product
    """
)
df[['description', 'product']]

INFO:root:: Wrangling :: remove_words :: description >> product


,description,product
0,Steel Blue Bottle,Bottle
1,Aluminum Red Can,Can
2,Rubber Yellow Tire,Tire
3,Titanium Blue Pipe,Pipe


---
## 9. Migration guide

### Do I need to change my existing recipes?

| Scenario | Action needed |
|---|---|
| You were using `tokenize_to_remove: True` explicitly | None — behaviour unchanged |
| You were using `tokenize_to_remove: False` explicitly | None — behaviour unchanged |
| You were relying on `tokenize_to_remove` defaulting to **`False`** (phrase matching) | Add `tokenize_to_remove: False` to keep the old behaviour |
| You were relying on **exact punctuation** matching (e.g. `A-100` must NOT match `A100`) | Add `characters_to_consider: all` |

### The most common change needed

If a `to_remove` column holds **multi-word phrases** (strings with spaces) and you need the **whole phrase** matched in the text, set `tokenize_to_remove: False`:

In [16]:
# BEFORE (relied on tokenize_to_remove=False default)
# NOW needs explicit tokenize_to_remove: False to preserve phrase matching
df = run(
    {
        'col': ['Brand Round Crown 3/8" x 7/16" Pack'],
        'phrases': ['Round Crown'],   # multi-word phrase in a string column
    },
    """
    - remove_words:
        input: col
        to_remove:
          - phrases
        output: out
        tokenize_to_remove: False     # <-- add this if you relied on the old default
    """
)
df[['col', 'out']]  # 'Round Crown' removed as a phrase

INFO:root:: Wrangling :: remove_words :: col >> out


,col,out
0,"Brand Round Crown 3/8"" x 7/16"" Pack","Brand 3/8"" x 7/16"" Pack"
